# Mature-series baseline benchmark

Evaluate the three baselines used to calibrate maturity in `02_01`. A product-store series enters an origin only when its history strictly before that origin satisfies the shared maturity rule.

The primary benchmark is the same-weekday moving average over the last four matching weekdays, with the recent 28-observation daily mean as fallback.

In [1]:
from pathlib import Path
import sys

import pandas as pd
from IPython.display import display

PROJECT_ROOT = next(
    (path.resolve() for path in [Path('../..'), Path('..'), Path('.')]
     if (path / 'src').exists()),
    None,
)
if PROJECT_ROOT is None:
    raise FileNotFoundError('Could not find the project root containing src/.')
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from src.models.benchmark import (
    MODEL_COLUMNS,
    PRIMARY_MODEL,
    load_benchmark_design,
    run_benchmark,
    segment_wape,
    summarize_models,
)

pd.set_option('display.max_columns', None)
pd.set_option('display.width', 180)

## Design and origin population

In [2]:
design = load_benchmark_design()
design_table = pd.DataFrame([{
    'minimum active days': design.min_active_days,
    'minimum demand days': design.min_demand_days,
    'first origin': design.first_origin.date(),
    'origin spacing days': design.origin_spacing_days,
    'forecast horizon days': design.forecast_horizon_days,
}])
display(design_table)

result = run_benchmark(design=design)
display(result.data_audit)
print(f'Number of evaluated origins: {len(result.origin_summary):,}')
display(result.origin_summary.style.format({
    'known_series': '{:,.0f}',
    'mature_series': '{:,.0f}',
    'mature_fcm_series': '{:,.0f}',
    'mature_pseudo_series': '{:,.0f}',
    'evaluated_series': '{:,.0f}',
}))

,minimum active days,minimum demand days,first origin,origin spacing days,forecast horizon days
0,61,10,2025-12-01,28,7


,source_rows,series,first_date,last_date,overlapping_source_rows,null_active_flags,active_rows_with_closure_reason,closed_rows_without_reason,closed_rows_with_demand,changing_source_series,changing_category_series
0,23993855,25169,2023-07-22,2026-07-21,0.0,0.0,0.0,0.0,0.0,0,0


Number of evaluated origins: 9


,origin,known_series,mature_series,mature_fcm_series,mature_pseudo_series,evaluated_series
0,2025-12-01 00:00:00,"24,259","23,782","1,801","21,981","23,782"
1,2025-12-29 00:00:00,"24,387","23,975","1,922","22,053","23,975"
2,2026-01-26 00:00:00,"24,704","24,075","1,986","22,089","24,075"
3,2026-02-23 00:00:00,"25,095","24,116","2,016","22,100","24,116"
4,2026-03-23 00:00:00,"25,101","24,318","2,130","22,188","24,318"
5,2026-04-20 00:00:00,"25,122","24,869","2,657","22,212","24,869"
6,2026-05-18 00:00:00,"25,129","25,002","2,749","22,253","25,002"
7,2026-06-15 00:00:00,"25,140","25,052","2,791","22,261","25,052"
8,2026-07-13 00:00:00,"25,169","25,099","2,837","22,262","25,099"


Each origin is evaluated independently. Counts can increase over time because a series is admitted from its first mature origin onward; no future maturity status is carried backward.

In [3]:
model_summary = summarize_models(result.forecasts)
model_summary.insert(1, 'model_label', model_summary['model'].map(MODEL_COLUMNS))
model_summary.insert(2, 'primary', model_summary['model'].eq(PRIMARY_MODEL))
display(model_summary.style.format({
    'pooled_wape': '{:.2%}',
    'relative_bias': '{:+.2%}',
    'forecast_to_actual_ratio': '{:.3f}',
    'median_series_wape': '{:.2%}',
    'seasonal_mase': '{:.3f}',
    'mae_kg': '{:.3f}',
    'mean_bias_kg': '{:+.3f}',
    'actual_kg': '{:,.1f}',
    'forecast_kg': '{:,.1f}',
    'n_series': '{:,.0f}',
    'n_origins': '{:,.0f}',
    'n_forecast_rows': '{:,.0f}',
    'seasonal_mase_valid_share': '{:.1%}',
}))

,model,model_label,primary,pooled_wape,relative_bias,forecast_to_actual_ratio,median_series_wape,seasonal_mase,mae_kg,mean_bias_kg,actual_kg,forecast_kg,n_series,n_origins,n_forecast_rows,seasonal_mase_valid_share
0,recent_mean,Recent mean (28 observations),False,99.04%,-19.68%,0.803,130.55%,0.882,0.881,-0.175,"1,156,111.7","928,609.7","25,099",9,"1,300,152",100.0%
1,same_weekday_moving_average,Same-weekday moving average (4 occurrences; recent-mean fallback),True,103.76%,-2.75%,0.973,138.53%,0.924,0.923,-0.024,"1,156,111.7","1,124,371.9","25,099",9,"1,300,152",100.0%
2,occurrence_x_positive_quantity,Occurrence x positive quantity,False,111.49%,-2.44%,0.976,141.74%,0.956,0.991,-0.022,"1,156,111.7","1,127,917.0","25,099",9,"1,300,152",100.0%


## Primary

Pooled WAPE and volume calibration are the primary benchmark readout. Relative bias is `(forecast - actual) / actual`; the forecast-to-actual ratio is the same calibration expressed around 1.

In [4]:
primary = model_summary.loc[
    model_summary['model'].eq(PRIMARY_MODEL),
    [
        'model_label',
        'pooled_wape',
        'relative_bias',
        'forecast_to_actual_ratio',
        'actual_kg',
        'forecast_kg',
        'n_series',
        'n_origins',
    ],
]
display(primary.style.format({
    'pooled_wape': '{:.2%}',
    'relative_bias': '{:+.2%}',
    'forecast_to_actual_ratio': '{:.3f}',
    'actual_kg': '{:,.1f}',
    'forecast_kg': '{:,.1f}',
    'n_series': '{:,.0f}',
    'n_origins': '{:,.0f}',
}))

,model_label,pooled_wape,relative_bias,forecast_to_actual_ratio,actual_kg,forecast_kg,n_series,n_origins
1,Same-weekday moving average (4 occurrences; recent-mean fallback),103.76%,-2.75%,0.973,"1,156,111.7","1,124,371.9","25,099",9


## Secondary

The secondary view describes typical-series error, scale-free weekly-seasonal error, error in kilograms, and pooled performance within operational segments. Seasonal MASE uses the pre-origin mean absolute difference at a seven-calendar-day lag.

In [5]:
secondary = model_summary.loc[
    model_summary['model'].eq(PRIMARY_MODEL),
    [
        'model_label',
        'median_series_wape',
        'seasonal_mase',
        'mae_kg',
        'mean_bias_kg',
        'seasonal_mase_valid_share',
    ],
]
display(secondary.style.format({
    'median_series_wape': '{:.2%}',
    'seasonal_mase': '{:.3f}',
    'mae_kg': '{:.3f}',
    'mean_bias_kg': '{:+.3f}',
    'seasonal_mase_valid_share': '{:.1%}',
}))

,model_label,median_series_wape,seasonal_mase,mae_kg,mean_bias_kg,seasonal_mase_valid_share
1,Same-weekday moving average (4 occurrences; recent-mean fallback),138.53%,0.924,0.923,-0.024,100.0%


In [6]:
primary_forecasts = result.forecasts[result.forecasts['model'].eq(PRIMARY_MODEL) & result.forecasts['is_active']].copy()
segment_format = {
    'pooled_wape': '{:.2%}',
    'relative_bias': '{:+.2%}',
    'forecast_to_actual_ratio': '{:.3f}',
    'actual_kg': '{:,.1f}',
    'n_series': '{:,.0f}',
    'n_origins': '{:,.0f}',
}
print('Segment-level WAPE by sourcing group')
display(segment_wape(primary_forecasts, 'sourcing_group').style.format(segment_format))
print('Segment-level WAPE by merchandise category')
display(segment_wape(primary_forecasts, 'category_id').style.format(segment_format))
print('All-model results by sourcing group and merchandise category')
combined_segments = segment_wape(
    result.forecasts[result.forecasts['is_active']], ['sourcing_group', 'category_id']
)
combined_segments.insert(
    3, 'model_label', combined_segments['model'].map(MODEL_COLUMNS)
)
display(combined_segments.style.format(segment_format))

Segment-level WAPE by sourcing group


,sourcing_group,model,pooled_wape,relative_bias,forecast_to_actual_ratio,actual_kg,n_series,n_origins
0,FCM,same_weekday_moving_average,100.84%,+1.76%,1.018,"23,544.2","2,837",9
1,Pseudo,same_weekday_moving_average,103.82%,-2.84%,0.972,"1,132,567.5","22,262",9


Segment-level WAPE by merchandise category


,category_id,model,pooled_wape,relative_bias,forecast_to_actual_ratio,actual_kg,n_series,n_origins
0,890,same_weekday_moving_average,103.84%,-2.84%,0.972,"1,132,230.5","22,299",9
1,900,same_weekday_moving_average,100.28%,+1.82%,1.018,"23,881.2","2,800",9


All-model results by sourcing group and merchandise category


,sourcing_group,category_id,model,model_label,pooled_wape,relative_bias,forecast_to_actual_ratio,actual_kg,n_series,n_origins
0,FCM,890,occurrence_x_positive_quantity,Occurrence x positive quantity,458.73%,+267.45%,3.674,37.4,98,9
1,FCM,890,recent_mean,Recent mean (28 observations),323.95%,+130.47%,2.305,37.4,98,9
2,FCM,890,same_weekday_moving_average,Same-weekday moving average (4 occurrences; recent-mean fallback),370.12%,+175.20%,2.752,37.4,98,9
3,FCM,900,occurrence_x_positive_quantity,Occurrence x positive quantity,99.71%,+2.42%,1.024,"23,506.8","2,739",9
4,FCM,900,recent_mean,Recent mean (28 observations),93.83%,-16.00%,0.840,"23,506.8","2,739",9
5,FCM,900,same_weekday_moving_average,Same-weekday moving average (4 occurrences; recent-mean fallback),100.41%,+1.48%,1.015,"23,506.8","2,739",9
6,Pseudo,890,occurrence_x_positive_quantity,Occurrence x positive quantity,111.73%,-2.56%,0.974,"1,132,193.1","22,201",9
7,Pseudo,890,recent_mean,Recent mean (28 observations),99.14%,-19.77%,0.802,"1,132,193.1","22,201",9
8,Pseudo,890,same_weekday_moving_average,Same-weekday moving average (4 occurrences; recent-mean fallback),103.83%,-2.85%,0.972,"1,132,193.1","22,201",9
9,Pseudo,900,occurrence_x_positive_quantity,Occurrence x positive quantity,93.56%,+25.37%,1.254,374.4,61,9
